# Khám phá dữ liệu và kiểm tra chất lượng

Notebook phân tích 12 CSV PRSA theo trạm. Dữ liệu nguồn và CSV chuẩn hóa được giữ nguyên; phần so sánh trước/sau ở mục 4 chỉ tạo bản tạm cho trực quan hóa theo đúng pipeline trong `src/features.py` (forward-fill tối đa 6 giờ, hướng gió `Unknown`, median học từ train/validation trước mốc test). Target PM2.5 không được impute; giá trị cao không bị loại. Mục tiêu dự kiến là PM2.5 của cùng trạm ở giờ kế tiếp (t+1). Các CSV OHLCV và ảnh JPG ở cấp `data/raw/` không thuộc bộ PRSA nên không được nạp.

## 1. Nạp dữ liệu và kiểm tra cấu trúc

Tự tìm thư mục gốc từ working directory hoặc các thư mục cha để notebook chạy được từ repository root hoặc `notebooks/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelcolor": "#334155",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "grid.alpha": 0.22,
})


def style_axes(axes):
    """Apply consistent grid styling to a sequence of axes."""
    for ax in axes:
        ax.grid(axis="y", alpha=0.22)
        ax.set_axisbelow(True)


data_dir = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    possible = candidate / "data" / "raw" / "PRSA_Data_20130301-20170228"
    if possible.is_dir():
        data_dir = possible
        project_root = candidate
        break

if data_dir is None:
    raise FileNotFoundError("Không tìm thấy thư mục CSV PRSA. Hãy chạy notebook bên trong repository.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

csv_paths = sorted(data_dir.glob("PRSA_Data_*_20130301-20170228.csv"))
if len(csv_paths) != 12:
    raise ValueError(f"Cần đúng 12 CSV PRSA, tìm thấy {len(csv_paths)} tại {data_dir}")

frames = []
for path in csv_paths:
    frame = pd.read_csv(path, na_values=["NA"])
    frame["source_file"] = path.name
    frames.append(frame)
df = pd.concat(frames, ignore_index=True)

required_columns = {
    "No", "year", "month", "day", "hour", "PM2.5", "PM10", "SO2", "NO2",
    "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM", "station"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Thiếu các cột bắt buộc: {sorted(missing_columns)}")

df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]], errors="coerce")
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)
stations = sorted(df["station"].dropna().unique())
station_palette = dict(zip(stations, sns.color_palette("husl", n_colors=len(stations))))

print(f"Project root: {project_root}")
print(f"Số tệp: {len(csv_paths)} | Kích thước gộp: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Các trạm: {stations}")
print(f"Khoảng thời gian: {df['datetime'].min()} → {df['datetime'].max()}")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

## 2. Chất lượng và tính liên tục của dữ liệu

Kiểm tra số dòng, số timestamp duy nhất, timestamp trùng/không hợp lệ và số giờ khuyết theo trạm. Thống kê missing bao gồm riêng target PM2.5; chưa impute hoặc loại dòng.

In [ ]:
expected_start = pd.Timestamp("2013-03-01 00:00:00")
expected_end = pd.Timestamp("2017-02-28 23:00:00")
expected_hours = pd.date_range(expected_start, expected_end, freq="h")

time_checks = []
for station, group in df.groupby("station", sort=True):
    valid_times = group["datetime"].dropna()
    unique_times = pd.DatetimeIndex(valid_times.unique()).sort_values()
    time_checks.append({
        "station": station,
        "rows": len(group),
        "unique_timestamps": len(unique_times),
        "duplicate_timestamps": int(valid_times.duplicated().sum()),
        "invalid_timestamps": int(group["datetime"].isna().sum()),
        "missing_hours_in_expected_range": len(expected_hours.difference(unique_times)),
        "start": unique_times.min() if len(unique_times) else pd.NaT,
        "end": unique_times.max() if len(unique_times) else pd.NaT,
    })
time_checks = pd.DataFrame(time_checks).set_index("station")
display(time_checks)
print(f"Số giờ kỳ vọng mỗi trạm: {len(expected_hours):,}")

analysis_columns = [column for column in df.columns if column != "source_file"]
missing_summary = pd.DataFrame({
    "missing_count": df[analysis_columns].isna().sum(),
    "missing_percent": df[analysis_columns].isna().mean().mul(100),
}).sort_values("missing_percent", ascending=False)
display(missing_summary)

missing_by_station = df.groupby("station")[analysis_columns].apply(
    lambda group: group.isna().mean().mul(100)
)
display(missing_by_station["PM2.5"].rename("PM2.5_missing_percent").to_frame())
display(missing_by_station.T)

source_features = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM", "wd"
]
raw_missing_pct = df[source_features].isna().mean().mul(100).sort_values()
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=raw_missing_pct.values, y=raw_missing_pct.index, color="#3B82A0", ax=ax)
ax.set(title="Tỷ lệ giá trị thiếu theo đặc trưng (dữ liệu gốc)", xlabel="Thiếu (%)", ylabel="")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=4, fontsize=9)
ax.set_xlim(0, max(raw_missing_pct.max() * 1.18, 1))
style_axes([ax])
plt.tight_layout()
plt.show()

## 3. Phân bố PM2.5 và so sánh giữa trạm

Báo cáo các phân vị của PM2.5 và vẽ histogram/boxplot. Không loại bỏ giá trị cao chỉ vì chúng là ngoại lệ trên biểu đồ; cần xác minh ngữ cảnh trước khi kết luận.

## 4. So sánh trước và sau xử lý đầu vào

Các biểu đồ dưới đây đối chiếu dữ liệu gốc với bản tạm đi qua các bước trong `src/features.py`: forward-fill tối đa 6 giờ riêng từng trạm, `wd` còn thiếu thành `Unknown`, sau đó median imputation cho feature số. Median chỉ được fit trên phần train/validation trước mốc test; dữ liệu test chỉ được transform. Để giữ đơn vị µg/m³ dễ so sánh, biểu đồ tắt scaling; Linear Regression ở notebook 02 vẫn dùng `scale=True`. Đây là minh họa trực quan, không phải đánh giá mô hình. Target PM2.5 không được điền; các giá trị cao vẫn được giữ nguyên. CSV chuẩn hóa và `df` gốc không bị thay đổi.

In [ ]:
import numpy as np

from src.data_loader import time_split
from src.features import NUMERIC_SOURCE_FEATURES, add_features, build_preprocessor, split_xy

featured_data = add_features(df)
labeled_data = featured_data.dropna(subset=["target"]).reset_index(drop=True)
train_validation, test_data = time_split(labeled_data)
X_train_validation, _ = split_xy(train_validation)
X_all, _ = split_xy(featured_data)

visual_preprocessor = build_preprocessor(scale=False)
visual_preprocessor.fit(X_train_validation)
transformed_all = visual_preprocessor.transform(X_all)
transformed_names = list(visual_preprocessor.get_feature_names_out())

source_columns = [*NUMERIC_SOURCE_FEATURES, "wd"]
processed_source = pd.DataFrame(index=featured_data.index)
for column in NUMERIC_SOURCE_FEATURES:
    output_column = f"numeric__{column}"
    output_values = transformed_all[:, transformed_names.index(output_column)]
    if hasattr(output_values, "toarray"):
        output_values = output_values.toarray().ravel()
    else:
        output_values = np.asarray(output_values).ravel()
    processed_source[column] = output_values
processed_source["wd"] = featured_data["wd"].astype("string").to_numpy()

before_source = df[source_columns].reset_index(drop=True)
after_source = processed_source[source_columns]
missing_comparison = pd.DataFrame({
    "Trước xử lý": before_source.isna().mean().mul(100),
    "Sau xử lý": after_source.isna().mean().mul(100),
}).rename_axis("Đặc trưng").reset_index().melt(
    id_vars="Đặc trưng", var_name="Giai đoạn", value_name="Thiếu (%)"
)

fig, axes = plt.subplots(1, 2, figsize=(19, 7), gridspec_kw={"width_ratios": [1, 1.8]})
sns.barplot(
    data=missing_comparison,
    x="Thiếu (%)",
    y="Đặc trưng",
    hue="Giai đoạn",
    palette={"Trước xử lý": "#D9826B", "Sau xử lý": "#3B9B83"},
    ax=axes[0],
)
axes[0].set(title="Tỷ lệ thiếu của feature", xlabel="Thiếu (%)", ylabel="")
axes[0].legend(title="")

pm25_before = df[["station", "PM2.5"]].dropna().assign(giai_doan="Trước xử lý")
pm25_after = after_source[["PM2.5"]].assign(
    station=featured_data["station"].to_numpy(), giai_doan="Sau xử lý"
)
pm25_comparison = pd.concat([pm25_before, pm25_after], ignore_index=True)
pm25_plot_limit = pm25_comparison["PM2.5"].quantile(0.995)
sns.boxplot(
    data=pm25_comparison,
    x="station",
    y="PM2.5",
    hue="giai_doan",
    order=stations,
    showfliers=False,
    palette={"Trước xử lý": "#D9826B", "Sau xử lý": "#3B9B83"},
    width=0.75,
    ax=axes[1],
)
axes[1].set(
    title="Phân bố PM2.5 đầu vào theo trạm",
    xlabel="Trạm quan trắc",
    ylabel="PM2.5 (µg/m³)",
)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylim(0, pm25_plot_limit)
axes[1].legend(title="")
style_axes(axes)
fig.suptitle("So sánh dữ liệu trước và sau xử lý feature", fontsize=16, fontweight="bold", y=1.02)
fig.text(
    0.5, -0.02,
    f"Giới hạn trục PM2.5 tại P99,5 = {pm25_plot_limit:.1f} chỉ để dễ đọc; giá trị cao vẫn được giữ. "
    "Target không impute; median fit trên train/validation trước mốc test.",
    ha="center", fontsize=9, color="#475569",
)
plt.tight_layout()
plt.show()

display(missing_comparison.pivot(index="Đặc trưng", columns="Giai đoạn", values="Thiếu (%)").round(2))
print(f"Số dòng dùng fit imputer: {len(X_train_validation):,}")
print(f"Số dòng test chỉ transform: {len(test_data):,}")
print(f"Target PM2.5 vẫn thiếu (không impute): {int(featured_data['target'].isna().sum()):,}")
print(f"PM2.5 = 999 vẫn còn trong dữ liệu nguồn: {int(df['PM2.5'].eq(999).sum()):,}")

In [ ]:
pm25_summary = df.groupby("station")["PM2.5"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
)
display(pm25_summary)

plot_limit = df["PM2.5"].quantile(0.995)
fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={"width_ratios": [1, 1.7]})
sns.histplot(
    data=df.loc[df["PM2.5"].le(plot_limit)],
    x="PM2.5",
    bins=55,
    color="#2878A5",
    edgecolor="white",
    ax=axes[0],
)
axes[0].set(
    title="Phân bố PM2.5 (đến phân vị 99,5%)",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Số quan sát",
)
sns.boxplot(
    data=df,
    x="station",
    y="PM2.5",
    order=stations,
    showfliers=False,
    color="#7DB7A8",
    width=0.65,
    ax=axes[1],
)
axes[1].set(
    title="So sánh PM2.5 theo trạm (ẩn điểm ngoại lệ để dễ đọc)",
    xlabel="Trạm quan trắc",
    ylabel="PM2.5 (µg/m³)",
)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylim(0, plot_limit)
style_axes(axes)
fig.suptitle("Phân bố PM2.5 trong dữ liệu gốc", fontsize=16, fontweight="bold", y=1.03)
fig.text(
    0.5, -0.02,
    f"Chỉ giới hạn trục để dễ quan sát (P99,5 = {plot_limit:.1f}); không xóa hay thay đổi giá trị nguồn.",
    ha="center", fontsize=9, color="#475569",
)
plt.tight_layout()
plt.show()

## 5. Xu hướng theo thời gian, giờ và mùa

Dùng trung bình nhóm để trực quan hóa xu hướng mà không vẽ từng điểm trong hơn 400 nghìn dòng. Mùa được gom theo tháng khí tượng học; các đường dùng màu nhất quán theo trạm.

In [ ]:
monthly_pm25 = (
    df.dropna(subset=["datetime"])
    .set_index("datetime")
    .groupby("station")["PM2.5"]
    .resample("MS")
    .mean()
    .rename("PM2.5")
    .reset_index()
)
fig, ax = plt.subplots(figsize=(17, 6))
sns.lineplot(
    data=monthly_pm25, x="datetime", y="PM2.5", hue="station",
    hue_order=stations, palette=station_palette, linewidth=1.5, ax=ax,
)
ax.set(title="PM2.5 trung bình theo tháng", xlabel="Thời gian", ylabel="PM2.5 trung bình (µg/m³)")
ax.legend(title="Trạm", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
style_axes([ax])
plt.tight_layout()
plt.show()

hourly_pm25 = df.groupby(["station", "hour"], as_index=False)["PM2.5"].mean()
month_to_season = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}
df["season"] = df["month"].map(month_to_season)
seasonal_pm25 = df.groupby(["station", "season"], as_index=False)["PM2.5"].mean()
season_order = ["Spring", "Summer", "Autumn", "Winter"]

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={"width_ratios": [1.35, 1]})
sns.lineplot(
    data=hourly_pm25, x="hour", y="PM2.5", hue="station",
    hue_order=stations, palette=station_palette, linewidth=1.7, ax=axes[0],
)
axes[0].set(title="PM2.5 trung bình theo giờ", xlabel="Giờ trong ngày", ylabel="PM2.5 (µg/m³)")
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(title="Trạm", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)

seasonal_matrix = seasonal_pm25.pivot(index="station", columns="season", values="PM2.5").reindex(
    index=stations, columns=season_order
)
sns.heatmap(
    seasonal_matrix, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5,
    linecolor="white", cbar_kws={"label": "PM2.5 trung bình (µg/m³)"}, ax=axes[1],
)
axes[1].set(title="PM2.5 theo mùa và trạm", xlabel="Mùa", ylabel="Trạm quan trắc")
style_axes(axes)
plt.tight_layout()
plt.show()
display(seasonal_matrix.round(2))

## 6. Tương quan và các phát hiện cần ghi nhận

Tương quan Pearson thể hiện liên hệ tuyến tính, không chứng minh quan hệ nhân quả. Cột `No` là số thứ tự dòng nên không đưa vào ma trận.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.drop("No", errors="ignore")
correlation = df[numeric_columns].corr()
display(correlation[["PM2.5"]].sort_values("PM2.5", ascending=False))

mask = np.triu(np.ones_like(correlation, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    correlation, mask=mask, cmap="vlag", vmin=-1, vmax=1, center=0,
    annot=True, fmt=".2f", annot_kws={"size": 8}, linewidths=0.4, square=True,
    cbar_kws={"shrink": 0.8, "label": "Hệ số tương quan Pearson"}, ax=ax,
)
ax.set_title("Tương quan Pearson giữa các biến số", pad=16)
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

quality_summary = pd.Series({
    "Số tệp CSV": len(csv_paths),
    "Số trạm": df["station"].nunique(),
    "Tổng số dòng": len(df),
    "Số dòng kỳ vọng (12 × 35.064)": 12 * len(expected_hours),
    "Dòng trùng station-datetime": int(df.duplicated(["station", "datetime"]).sum()),
    "Timestamp không parse được": int(df["datetime"].isna().sum()),
    "PM2.5 missing (%)": round(df["PM2.5"].isna().mean() * 100, 3),
    "PM2.5 nhỏ hơn 0": int(df["PM2.5"].lt(0).sum()),
    "Tổng số ô missing": int(df.isna().sum().sum()),
}, name="Giá trị")
display(quality_summary.to_frame())

print("Ghi chú:")
print("- Không dùng dòng thiếu PM2.5 làm target huấn luyện hoặc đánh giá.")
print("- Imputer chỉ được fit trên train; các fold CV fit riêng trong từng fold.")
print("- Lag/rolling cho dự đoán t+1 chỉ dùng quá khứ và tính riêng theo trạm.")
print("- Không loại PM2.5 cao chỉ dựa trên boxplot; cần xác minh đó là lỗi đo hay sự kiện thực tế.")

## 7. Lưu bản dữ liệu chuẩn hóa

Bước này chỉ gộp 12 trạm, thêm `datetime` và sắp xếp bản sao dữ liệu nguồn để lưu trong `data/processed/`. Bản CSV này vẫn giữ missing và các giá trị PM2.5 cao; phần so sánh trước/sau ở mục 4 không ghi đè hay thay đổi dữ liệu này.

In [ ]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
processed_path = processed_dir / "beijing_air_quality_standardized.csv"

# Keep source columns, add datetime, and exclude EDA-only helper columns.
standardized_columns = [column for column in df.columns if column not in {"source_file", "season"}]
standardized_df = df[standardized_columns].sort_values(["station", "datetime"]).reset_index(drop=True)
standardized_df.to_csv(processed_path, index=False, date_format="%Y-%m-%d %H:%M:%S")

# Reload the output and verify the data properties we intend to preserve.
saved_df = pd.read_csv(processed_path, na_values=["NA"], parse_dates=["datetime"])
assert len(saved_df) == 12 * len(expected_hours)
assert saved_df["station"].nunique() == 12
assert not saved_df.duplicated(["station", "datetime"]).any()
assert saved_df["datetime"].min() == expected_start
assert saved_df["datetime"].max() == expected_end
assert saved_df["PM2.5"].isna().sum() == df["PM2.5"].isna().sum()
assert saved_df["PM2.5"].eq(999).sum() == df["PM2.5"].eq(999).sum()
assert saved_df.isna().sum().sum() == df[standardized_columns].isna().sum().sum()

print(f"Saved standardized data: {processed_path.relative_to(project_root)}")
print(f"Rows x columns: {saved_df.shape[0]:,} x {saved_df.shape[1]}")
print(f"Missing cells preserved: {int(saved_df.isna().sum().sum()):,}")
print(f"Missing PM2.5 preserved: {int(saved_df['PM2.5'].isna().sum()):,}")
print(f"PM2.5 values >= 500 retained: {int(saved_df['PM2.5'].ge(500).sum()):,}")
print(f"PM2.5 value 999 retained: {int(saved_df['PM2.5'].eq(999).sum())}")